# CS5224 Data Curation — Stage 2A

## Category mapping and pilot-category selection

Stage 1E established:

- catalog eligibility;
- current stock availability;
- active offer eligibility;
- brand / price / specification quality flags.

Stage 2A maps the retailer source categories into the eight agreed CartOptix catalogue categories:

1. **Rice & Noodles**
2. **Drinks**
3. **Dairy**
4. **Breakfast Foods**
5. **Canned & Packaged Food**
6. **Snacks**
7. **Household Cleaning**
8. **Toiletries & Personal Care**

### Design principle

Stage 2A is conservative.

- High-confidence mapping is performed from `source_category_normalized`.
- Product-name rules are used only to create **review suggestions** when the source category is not sufficient.
- Product-name suggestions are **not automatically accepted** into the final category.
- Conflicts and unmapped rows are retained for review.
- No Stage 1E fields are changed.

### Pilot selection

The notebook selects up to three pilot categories using **balanced cross-retailer coverage** among clean catalog-eligible records.

The pilot pool includes both:

- currently in-stock records; and
- catalog-eligible out-of-stock records.

Availability remains separate through `currently_available` and `offer_eligible`.


## 1. Imports and project paths

In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().parent
STAGING_DATA_DIR = PROJECT_ROOT / "data" / "staging"
CURATED_DATA_DIR = PROJECT_ROOT / "data" / "curated"

CURATED_DATA_DIR.mkdir(
    parents=True,
    exist_ok=True
)

STAGE1E_FILE = (
    STAGING_DATA_DIR
    / "all_retailers_stage1e_quality.csv"
)

print("Current notebook folder:")
print(Path.cwd())

print("\nProject root:")
print(PROJECT_ROOT)

print("\nStage 1E input:")
print(STAGE1E_FILE)

print("\nExists:", STAGE1E_FILE.exists())

assert STAGE1E_FILE.exists(), (
    "Stage 1E checkpoint not found. "
    "Run the revised Data_Curation_Stage1E notebook first."
)


Current notebook folder:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\scripts

Project root:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix

Stage 1E input:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\all_retailers_stage1e_quality.csv

Exists: True


## 2. Define expected Stage 1E schema

In [2]:
STAGE1E_COLUMNS = [
    "retailer",
    "source_file",
    "source_sheet",
    "source_row_number",
    "source_record_id",

    "source_category_raw",
    "source_category_normalized",

    "product_name_raw",
    "product_name_normalized",

    "brand_raw",
    "brand_normalized",
    "brand_status",

    "specification_raw",
    "specification_normalized",
    "pack_count",
    "unit_size",
    "unit",
    "total_quantity",
    "total_quantity_base",
    "base_unit",
    "spec_parse_status",
    "spec_parse_pattern",
    "spec_parse_notes",
    "spec_quality_status",

    "product_price_raw",
    "product_price",
    "original_price_raw",
    "original_price",
    "price_status",

    "savings_raw",

    "stock_raw",
    "in_stock",
    "stock_status",
    "currently_available",

    "offer_raw",
    "has_offer",

    "image_url_raw",
    "product_url_raw",
    "comments_raw",

    "quality_issue_codes",
    "hard_exclusion_reason",
    "needs_manual_review",
    "basic_eligibility",
    "catalog_eligible",
    "offer_eligible",
    "eligibility_status",
]

EXPECTED_TOTAL_ROWS = 177_638

EXPECTED_RETAILER_COUNTS = {
    "redmart": 97_779,
    "fairprice": 69_402,
    "coldstorage": 10_457,
}

TARGET_CATEGORIES = [
    "Rice & Noodles",
    "Drinks",
    "Dairy",
    "Breakfast Foods",
    "Canned & Packaged Food",
    "Snacks",
    "Household Cleaning",
    "Toiletries & Personal Care",
]

assert len(TARGET_CATEGORIES) == 8
assert len(set(TARGET_CATEGORIES)) == 8


## 3. Load the Stage 1E checkpoint

In [3]:
stage1e = pd.read_csv(
    STAGE1E_FILE,
    dtype={
        "retailer": "string",
        "source_file": "string",
        "source_sheet": "string",
        "source_record_id": "string",
        "source_category_raw": "string",
        "source_category_normalized": "string",
        "product_name_raw": "string",
        "product_name_normalized": "string",
        "brand_raw": "string",
        "brand_normalized": "string",
        "brand_status": "string",
        "specification_raw": "string",
        "specification_normalized": "string",
        "unit": "string",
        "base_unit": "string",
        "spec_parse_status": "string",
        "spec_parse_pattern": "string",
        "spec_parse_notes": "string",
        "spec_quality_status": "string",
        "price_status": "string",
        "stock_raw": "string",
        "stock_status": "string",
        "offer_raw": "string",
        "image_url_raw": "string",
        "product_url_raw": "string",
        "comments_raw": "string",
        "quality_issue_codes": "string",
        "hard_exclusion_reason": "string",
        "eligibility_status": "string",
    }
)

# Restore nullable integer / numeric types.
stage1e["source_row_number"] = (
    pd.to_numeric(
        stage1e["source_row_number"],
        errors="coerce"
    )
    .astype("Int64")
)

stage1e["pack_count"] = (
    pd.to_numeric(
        stage1e["pack_count"],
        errors="coerce"
    )
    .astype("Int64")
)

for column in [
    "unit_size",
    "total_quantity",
    "total_quantity_base",
    "product_price",
    "original_price",
]:
    stage1e[column] = (
        pd.to_numeric(
            stage1e[column],
            errors="coerce"
        )
        .astype("Float64")
    )

def to_nullable_boolean(series):
    normalized = (
        series
        .astype("string")
        .str.strip()
        .str.casefold()
    )

    mapping = {
        "true": True,
        "false": False,
        "yes": True,
        "no": False,
        "1": True,
        "0": False,
    }

    return (
        normalized
        .map(mapping)
        .astype("boolean")
    )

for column in [
    "in_stock",
    "currently_available",
    "has_offer",
    "needs_manual_review",
    "basic_eligibility",
    "catalog_eligible",
    "offer_eligible",
]:
    stage1e[column] = to_nullable_boolean(
        stage1e[column]
    )

print("Loaded Stage 1E shape:")
print(stage1e.shape)

display(stage1e.head(3))


Loaded Stage 1E shape:
(177638, 46)


,retailer,source_file,source_sheet,source_row_number,source_record_id,source_category_raw,source_category_normalized,product_name_raw,product_name_normalized,brand_raw,...,image_url_raw,product_url_raw,comments_raw,quality_issue_codes,hard_exclusion_reason,needs_manual_review,basic_eligibility,catalog_eligible,offer_eligible,eligibility_status
0,redmart,redmart_all_products_final.xlsx,Sheet1,2,redmart_00000002,shop-chilled-desserts,shop-chilled-desserts,Sojasun Natural Soya Yoghurt,sojasun natural soya yoghurt,Sojasun,...,https://sg-live-01.slatic.net/p/5eaa6087b6c250...,https://www.lazada.sg/products/pdp-i303336401-...,<NA>,NONE,NONE,False,True,True,True,ELIGIBLE_PRELIMINARY
1,redmart,redmart_all_products_final.xlsx,Sheet1,3,redmart_00000003,shop-chilled-desserts,shop-chilled-desserts,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix chocolate custard dessert cream 4x125g,Monoprix,...,https://sg-test-11.slatic.net/p/9f86cfbce0b472...,https://www.lazada.sg/products/pdp-i462726209-...,<NA>,NONE,NONE,False,True,True,True,ELIGIBLE_PRELIMINARY
2,redmart,redmart_all_products_final.xlsx,Sheet1,4,redmart_00000004,shop-chilled-desserts,shop-chilled-desserts,Top Gourmet Cheng Tng,top gourmet cheng tng,Top Gourmet,...,https://sg-live-01.slatic.net/p/7579bbca2beaa0...,https://www.lazada.sg/products/pdp-i303250731-...,<NA>,NONE,NONE,False,True,True,True,ELIGIBLE_PRELIMINARY


## 4. Validate Stage 1E input

In [4]:
assert stage1e.columns.tolist() == STAGE1E_COLUMNS, (
    "Stage 1E columns do not match the expected revised schema."
)

assert len(stage1e) == EXPECTED_TOTAL_ROWS

assert (
    stage1e["source_record_id"]
    .isna()
    .sum()
    == 0
)

assert (
    stage1e["source_record_id"]
    .duplicated()
    .sum()
    == 0
)

for retailer, expected_count in EXPECTED_RETAILER_COUNTS.items():
    actual_count = (
        stage1e["retailer"] == retailer
    ).sum()

    assert actual_count == expected_count, (
        f"{retailer}: expected {expected_count:,}, "
        f"found {actual_count:,}."
    )

print("PASS: Stage 1E input validation succeeded.")


PASS: Stage 1E input validation succeeded.


## 5. Preserve the complete Stage 1E dataset

In [5]:
stage1e_snapshot = stage1e.copy(deep=True)

print(
    f"Stage 1E snapshot created: "
    f"{stage1e_snapshot.shape[0]:,} rows x "
    f"{stage1e_snapshot.shape[1]} columns"
)


Stage 1E snapshot created: 177,638 rows x 46 columns


## 6. Define conservative source-category mapping rules

These rules operate on `source_category_normalized`.

They intentionally cover **obvious source-category terminology only**.

A row is automatically mapped only if exactly one target-category rule matches its source category.

If multiple rules match, the row is marked as a conflict.

If no source-category rule matches, product-name rules may provide a review suggestion later.


In [6]:
SOURCE_CATEGORY_PATTERNS = {
    "Rice & Noodles": (
        r"\b(?:rice|noodles?|pasta|vermicelli|"
        r"bee\s*hoon|bihun|mee|macaroni|spaghetti)\b"
    ),

    "Drinks": (
        r"\b(?:drinks?|beverages?|water|juices?|coffee|tea|"
        r"soft\s*drinks?|carbonated|soda|isotonic|"
        r"energy\s*drinks?|sports\s*drinks?|cordials?)\b"
    ),

    "Dairy": (
        r"\b(?:dairy|milk|cheese|yogurt|yoghurt|butter|"
        r"cultured\s*milk)\b"
    ),

    "Breakfast Foods": (
        r"\b(?:breakfast|cereals?|oats?|granola|muesli|"
        r"corn\s*flakes?|cornflakes?)\b"
    ),

    "Canned & Packaged Food": (
        r"\b(?:canned|tinned|packaged|instant\s*food|"
        r"instant\s*meal|soups?|broths?|stocks?|"
        r"sauces?|condiments?|ready\s*meals?|"
        r"ready[-\s]*to[-\s]*eat)\b"
    ),

    "Snacks": (
        r"\b(?:snacks?|biscuits?|crackers?|chips?|crisps?|"
        r"chocolates?|confectionery|cand(?:y|ies)|sweets?|"
        r"popcorn|nuts?|desserts?|jell(?:y|ies)|"
        r"cakes?|cookies?)\b"
    ),

    "Household Cleaning": (
        r"\b(?:household\s*cleaning|cleaners?|cleaning|"
        r"laundry|detergents?|dishwashing|dish\s*wash|"
        r"bleach|floor\s*care|surface\s*cleaners?|"
        r"toilet\s*cleaners?|fabric\s*softeners?)\b"
    ),

    "Toiletries & Personal Care": (
        r"\b(?:toiletries?|personal\s*care|oral\s*care|"
        r"toothpaste|toothbrush(?:es)?|shampoo|conditioner|"
        r"body\s*wash|soap|deodorants?|skin\s*care|skincare|"
        r"hair\s*care|feminine\s*care|sanitary|menstrual)\b"
    ),
}

assert set(SOURCE_CATEGORY_PATTERNS) == set(TARGET_CATEGORIES)


## 7. Define product-name fallback rules

These rules are **suggestions only**.

They are used only when the source category does not match any high-confidence category rule.

A product-name suggestion is never automatically promoted to `target_category_final` in Stage 2A.


In [7]:
PRODUCT_NAME_PATTERNS = {
    "Rice & Noodles": (
        r"\b(?:rice|noodles?|pasta|vermicelli|"
        r"bee\s*hoon|bihun|macaroni|spaghetti)\b"
    ),

    "Drinks": (
        r"\b(?:mineral\s*water|drinking\s*water|sparkling\s*water|"
        r"juice|coffee|tea|cola|soda|isotonic|"
        r"energy\s*drink|sports\s*drink|beverage)\b"
    ),

    "Dairy": (
        r"\b(?:fresh\s*milk|uht\s*milk|milk\s*drink|cheese|"
        r"yogurt|yoghurt|butter)\b"
    ),

    "Breakfast Foods": (
        r"\b(?:breakfast\s*cereal|cereal|oats?|granola|"
        r"muesli|corn\s*flakes?|cornflakes?)\b"
    ),

    "Canned & Packaged Food": (
        r"\b(?:canned|tinned|instant\s*soup|instant\s*meal|"
        r"broth|stock|cooking\s*sauce|pasta\s*sauce|"
        r"ready[-\s]*to[-\s]*eat)\b"
    ),

    "Snacks": (
        r"\b(?:snack|biscuit|cracker|potato\s*chips?|crisps?|"
        r"chocolate|candy|popcorn|cookies?|jelly|"
        r"gummy|gummies)\b"
    ),

    "Household Cleaning": (
        r"\b(?:laundry\s*detergent|dishwashing|dish\s*wash|"
        r"surface\s*cleaner|floor\s*cleaner|toilet\s*cleaner|"
        r"bleach|fabric\s*softener)\b"
    ),

    "Toiletries & Personal Care": (
        r"\b(?:toothpaste|toothbrush|shampoo|conditioner|"
        r"body\s*wash|deodorant|facial\s*cleanser|"
        r"hand\s*soap|sanitary\s*pads?|tampons?)\b"
    ),
}

assert set(PRODUCT_NAME_PATTERNS) == set(TARGET_CATEGORIES)


## 8. Build source-category match matrix

In [8]:
stage2a = stage1e.copy()

source_text = (
    stage2a["source_category_normalized"]
    .fillna("")
    .astype("string")
)

source_match_matrix = pd.DataFrame(
    {
        category: source_text.str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
        for category, pattern
        in SOURCE_CATEGORY_PATTERNS.items()
    },
    index=stage2a.index,
)

stage2a["source_category_match_count"] = (
    source_match_matrix
    .sum(axis=1)
    .astype("Int64")
)

source_unique_category = (
    source_match_matrix
    .idxmax(axis=1)
    .where(
        stage2a["source_category_match_count"]
        == 1
    )
    .astype("string")
)

print("Source-category match counts:")
print(
    stage2a[
        "source_category_match_count"
    ].value_counts().sort_index()
)


Source-category match counts:
source_category_match_count
0    119722
1     55787
2      2129
Name: count, dtype: Int64


## 9. Build product-name fallback match matrix

In [9]:
product_text = (
    stage2a["product_name_normalized"]
    .fillna("")
    .astype("string")
)

product_match_matrix = pd.DataFrame(
    {
        category: product_text.str.contains(
            pattern,
            case=False,
            regex=True,
            na=False,
        )
        for category, pattern
        in PRODUCT_NAME_PATTERNS.items()
    },
    index=stage2a.index,
)

stage2a["product_name_match_count"] = (
    product_match_matrix
    .sum(axis=1)
    .astype("Int64")
)

product_unique_category = (
    product_match_matrix
    .idxmax(axis=1)
    .where(
        stage2a["product_name_match_count"]
        == 1
    )
    .astype("string")
)


## 10. Create mapping candidates and statuses

In [10]:
stage2a["target_category_suggested"] = pd.Series(
    pd.NA,
    index=stage2a.index,
    dtype="string",
)

stage2a["target_category_final"] = pd.Series(
    pd.NA,
    index=stage2a.index,
    dtype="string",
)

stage2a["category_mapping_status"] = pd.Series(
    "UNMAPPED_REVIEW",
    index=stage2a.index,
    dtype="string",
)

stage2a["category_mapping_confidence"] = pd.Series(
    "NONE",
    index=stage2a.index,
    dtype="string",
)

stage2a["category_mapping_method"] = pd.Series(
    "NONE",
    index=stage2a.index,
    dtype="string",
)

# ---------------------------------------------------------
# High-confidence source-category mapping
# ---------------------------------------------------------

source_unique_mask = (
    stage2a["source_category_match_count"]
    == 1
)

stage2a.loc[
    source_unique_mask,
    "target_category_suggested"
] = source_unique_category.loc[
    source_unique_mask
]

stage2a.loc[
    source_unique_mask,
    "target_category_final"
] = source_unique_category.loc[
    source_unique_mask
]

stage2a.loc[
    source_unique_mask,
    "category_mapping_status"
] = "AUTO_SOURCE_CATEGORY"

stage2a.loc[
    source_unique_mask,
    "category_mapping_confidence"
] = "HIGH"

stage2a.loc[
    source_unique_mask,
    "category_mapping_method"
] = "SOURCE_CATEGORY_RULE"

# ---------------------------------------------------------
# Source-category conflicts
# ---------------------------------------------------------

source_conflict_mask = (
    stage2a["source_category_match_count"]
    > 1
)

stage2a.loc[
    source_conflict_mask,
    "category_mapping_status"
] = "SOURCE_CATEGORY_CONFLICT"

stage2a.loc[
    source_conflict_mask,
    "category_mapping_confidence"
] = "REVIEW"

stage2a.loc[
    source_conflict_mask,
    "category_mapping_method"
] = "SOURCE_CATEGORY_RULE"

# ---------------------------------------------------------
# Product-name suggestion only when source category
# has no match.
# ---------------------------------------------------------

no_source_match_mask = (
    stage2a["source_category_match_count"]
    == 0
)

product_unique_mask = (
    no_source_match_mask
    &
    (
        stage2a["product_name_match_count"]
        == 1
    )
)

stage2a.loc[
    product_unique_mask,
    "target_category_suggested"
] = product_unique_category.loc[
    product_unique_mask
]

stage2a.loc[
    product_unique_mask,
    "category_mapping_status"
] = "PRODUCT_TEXT_SUGGESTION"

stage2a.loc[
    product_unique_mask,
    "category_mapping_confidence"
] = "REVIEW"

stage2a.loc[
    product_unique_mask,
    "category_mapping_method"
] = "PRODUCT_NAME_RULE"

product_conflict_mask = (
    no_source_match_mask
    &
    (
        stage2a["product_name_match_count"]
        > 1
    )
)

stage2a.loc[
    product_conflict_mask,
    "category_mapping_status"
] = "PRODUCT_TEXT_CONFLICT"

stage2a.loc[
    product_conflict_mask,
    "category_mapping_confidence"
] = "REVIEW"

stage2a.loc[
    product_conflict_mask,
    "category_mapping_method"
] = "PRODUCT_NAME_RULE"

print("Category mapping status:")
print(
    stage2a["category_mapping_status"]
    .value_counts(dropna=False)
)


Category mapping status:
category_mapping_status
UNMAPPED_REVIEW             109112
AUTO_SOURCE_CATEGORY         55787
PRODUCT_TEXT_SUGGESTION      10197
SOURCE_CATEGORY_CONFLICT      2129
PRODUCT_TEXT_CONFLICT          413
Name: count, dtype: Int64


## 11. Create readable candidate-category strings for review

In [11]:
def candidate_string(match_row):
    categories = [
        category
        for category, matched
        in match_row.items()
        if bool(matched)
    ]

    if not categories:
        return "NONE"

    return ";".join(categories)


source_candidate_strings = (
    source_match_matrix
    .apply(
        candidate_string,
        axis=1
    )
    .astype("string")
)

product_candidate_strings = (
    product_match_matrix
    .apply(
        candidate_string,
        axis=1
    )
    .astype("string")
)

stage2a["source_category_candidates"] = (
    source_candidate_strings
)

stage2a["product_name_candidates"] = (
    product_candidate_strings
)


## 12. Validate automatic category assignments

In [12]:
final_categories = set(
    stage2a[
        "target_category_final"
    ]
    .dropna()
    .unique()
)

unexpected_categories = (
    final_categories
    - set(TARGET_CATEGORIES)
)

print(
    "Automatically accepted target categories:"
)

for category in sorted(
    final_categories
):
    print("-", category)

print(
    "\nUnexpected categories:",
    unexpected_categories
)

assert not unexpected_categories

# Only high-confidence source-category mappings may
# populate target_category_final at Stage 2A.
assert (
    stage2a.loc[
        stage2a[
            "target_category_final"
        ].notna(),
        "category_mapping_status"
    ]
    == "AUTO_SOURCE_CATEGORY"
).all()

print(
    "\nPASS: Automatic final category assignments "
    "come only from unique high-confidence "
    "source-category rules."
)


Automatically accepted target categories:
- Breakfast Foods
- Canned & Packaged Food
- Dairy
- Drinks
- Household Cleaning
- Rice & Noodles
- Snacks
- Toiletries & Personal Care

Unexpected categories: set()

PASS: Automatic final category assignments come only from unique high-confidence source-category rules.


## 13. Show mapping coverage

In [13]:
total_rows = len(stage2a)

auto_mapped_count = int(
    (
        stage2a[
            "category_mapping_status"
        ]
        == "AUTO_SOURCE_CATEGORY"
    ).sum()
)

product_suggestion_count = int(
    (
        stage2a[
            "category_mapping_status"
        ]
        == "PRODUCT_TEXT_SUGGESTION"
    ).sum()
)

conflict_count = int(
    stage2a[
        "category_mapping_status"
    ].isin(
        [
            "SOURCE_CATEGORY_CONFLICT",
            "PRODUCT_TEXT_CONFLICT",
        ]
    ).sum()
)

unmapped_count = int(
    (
        stage2a[
            "category_mapping_status"
        ]
        == "UNMAPPED_REVIEW"
    ).sum()
)

print(
    f"Total rows                  : {total_rows:,}"
)

print(
    f"Auto source-category mapped : {auto_mapped_count:,}"
)

print(
    f"Product-text suggestions    : {product_suggestion_count:,}"
)

print(
    f"Mapping conflicts            : {conflict_count:,}"
)

print(
    f"Unmapped review              : {unmapped_count:,}"
)

print()

print(
    "Auto mapping coverage:",
    f"{auto_mapped_count / total_rows * 100:.2f}%"
)


Total rows                  : 177,638
Auto source-category mapped : 55,787
Product-text suggestions    : 10,197
Mapping conflicts            : 2,542
Unmapped review              : 109,112

Auto mapping coverage: 31.40%


## 14. Verify all Stage 1E fields remain unchanged

In [14]:
assert stage2a[
    STAGE1E_COLUMNS
].equals(stage1e_snapshot), (
    "At least one Stage 1E field changed during Stage 2A."
)

print(
    "PASS: All 46 Stage 1E fields remain unchanged."
)


PASS: All 46 Stage 1E fields remain unchanged.


## 15. Build source-category inventory

This table helps the team review source categories instead of manually inspecting 177,638 individual rows.


In [15]:
source_category_inventory = (
    stage2a
    .groupby(
        [
            "retailer",
            "source_category_raw",
            "source_category_normalized",
        ],
        dropna=False
    )
    .agg(
        total_rows=(
            "source_record_id",
            "size"
        ),
        catalog_eligible_rows=(
            "catalog_eligible",
            lambda s: int(
                s.fillna(False).sum()
            )
        ),
        offer_eligible_rows=(
            "offer_eligible",
            lambda s: int(
                s.fillna(False).sum()
            )
        ),
        out_of_stock_rows=(
            "stock_status",
            lambda s: int(
                (s == "OUT_OF_STOCK").sum()
            )
        ),
        auto_target_category=(
            "target_category_final",
            lambda s: (
                s.dropna().iloc[0]
                if len(s.dropna()) > 0
                else pd.NA
            )
        ),
        mapping_status=(
            "category_mapping_status",
            lambda s: (
                "AUTO_SOURCE_CATEGORY"
                if (
                    s
                    == "AUTO_SOURCE_CATEGORY"
                ).all()
                else "REVIEW"
            )
        ),
    )
    .reset_index()
    .sort_values(
        [
            "catalog_eligible_rows",
            "total_rows",
        ],
        ascending=False
    )
)

display(
    source_category_inventory.head(30)
)


,retailer,source_category_raw,source_category_normalized,total_rows,catalog_eligible_rows,offer_eligible_rows,out_of_stock_rows,auto_target_category,mapping_status
1226,redmart,Biscuits & Crackers,biscuits & crackers,1840,1770,1294,507,Snacks,AUTO_SOURCE_CATEGORY
1667,redmart,Seasonal Accessories,seasonal accessories,1805,1693,1265,449,<NA>,REVIEW
1431,redmart,Fresh Herbs & Spices,fresh herbs & spices,1640,1553,1486,67,<NA>,REVIEW
1214,redmart,Baking Needs & Sugar,baking needs & sugar,1533,1518,1180,340,<NA>,REVIEW
1167,redmart,Asian Sauces & Pastes,asian sauces & pastes,1400,1320,1320,0,Canned & Packaged Food,AUTO_SOURCE_CATEGORY
1596,redmart,"Nuts, Peas & Seeds","nuts, peas & seeds",1397,1289,1004,342,Snacks,AUTO_SOURCE_CATEGORY
1756,redmart,White Goods & Products,white goods & products,1246,1176,771,415,<NA>,REVIEW
1580,redmart,Multivitamin Supplements,multivitamin supplements,1201,1015,733,323,<NA>,REVIEW
1649,redmart,Red Products,red products,1073,996,995,2,<NA>,REVIEW
1307,redmart,Cooking Utensils,cooking utensils,1453,880,650,350,<NA>,REVIEW


## 16. Build category mapping review queue

In [16]:
mapping_review_queue = stage2a.loc[
    stage2a[
        "target_category_final"
    ].isna(),
    [
        "retailer",
        "source_record_id",
        "source_category_raw",
        "source_category_normalized",
        "product_name_raw",
        "brand_normalized",
        "specification_raw",
        "product_price",
        "stock_status",
        "catalog_eligible",
        "offer_eligible",
        "target_category_suggested",
        "source_category_candidates",
        "product_name_candidates",
        "category_mapping_status",
        "category_mapping_confidence",
        "category_mapping_method",
        "product_url_raw",
    ]
].copy()

print(
    "Category mapping review queue:",
    f"{len(mapping_review_queue):,}"
)

display(
    mapping_review_queue.head(30)
)


Category mapping review queue: 121,851


,retailer,source_record_id,source_category_raw,source_category_normalized,product_name_raw,brand_normalized,specification_raw,product_price,stock_status,catalog_eligible,offer_eligible,target_category_suggested,source_category_candidates,product_name_candidates,category_mapping_status,category_mapping_confidence,category_mapping_method,product_url_raw
120,redmart,redmart_00000122,Eggs,eggs,RedMart 15 Eggs 15 X 60G,redmart,15 × 60 g,4.64,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i301088929-...
121,redmart,redmart_00000123,Eggs,eggs,Dasoon Premium Eggs 15S (Keep Chilled),dasoon,15 × 60 g,5.14,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i301168091-...
122,redmart,redmart_00000124,Eggs,eggs,RedMart Fresh Local Eggs (Large) 15X60G,redmart,15 × 60 g,5.39,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i3229271520...
123,redmart,redmart_00000125,Eggs,eggs,Chew's Corn Rich Eggs - 10 X 50G,chew's,10 × 50 g,3.3,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i301138659-...
124,redmart,redmart_00000126,Eggs,eggs,Seng Choon 60 G Farm Fresh Eggs (Keep Chilled)...,seng,10 × 60 g,3.88,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i303288014-...
125,redmart,redmart_00000127,Eggs,eggs,Dasoon Kampong Eggs,dasoon,550 g,3.52,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i1144508661...
126,redmart,redmart_00000128,Eggs,eggs,Chew's Fresh Eggs With Vitamin E (Keep Chilled...,chew's,10 × 60 g,3.75,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i301104802-...
127,redmart,redmart_00000129,Eggs,eggs,Chew's Fresh Eggs With Organic Selenium And Vi...,chew's,10 × 55 g,4.9,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i301146639-...
128,redmart,redmart_00000130,Eggs,eggs,RedMart 10 Eggs (10x60g),redmart,10 × 60 g,3.1,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i301104882-...
129,redmart,redmart_00000131,Eggs,eggs,Chew's Newly Laid Fresh Eggs With Vitamin E (K...,chew's,10 × 40 g,3.1,IN_STOCK,True,True,<NA>,NONE,NONE,UNMAPPED_REVIEW,NONE,NONE,https://www.lazada.sg/products/pdp-i301136700-...


## 17. Build target-category coverage among auto-mapped rows

Two separate coverage measures are retained:

- `catalog_eligible`: searchable clean listings, including out-of-stock records;
- `offer_eligible`: clean and currently in-stock purchasing offers.


In [17]:
auto_mapped = stage2a.loc[
    stage2a[
        "target_category_final"
    ].notna()
].copy()

category_retailer_coverage = (
    auto_mapped
    .groupby(
        [
            "target_category_final",
            "retailer",
        ]
    )
    .agg(
        total_rows=(
            "source_record_id",
            "size"
        ),
        catalog_eligible_rows=(
            "catalog_eligible",
            lambda s: int(
                s.fillna(False).sum()
            )
        ),
        offer_eligible_rows=(
            "offer_eligible",
            lambda s: int(
                s.fillna(False).sum()
            )
        ),
        catalog_out_of_stock_rows=(
            "stock_status",
            lambda s: int(
                (s == "OUT_OF_STOCK").sum()
            )
        ),
    )
    .reset_index()
)

display(
    category_retailer_coverage
)


,target_category_final,retailer,total_rows,catalog_eligible_rows,offer_eligible_rows,catalog_out_of_stock_rows
0,Breakfast Foods,coldstorage,275,248,219,41
1,Breakfast Foods,fairprice,351,333,314,29
2,Breakfast Foods,redmart,1014,951,635,340
3,Canned & Packaged Food,coldstorage,666,628,561,71
4,Canned & Packaged Food,fairprice,1465,1357,1265,176
5,Canned & Packaged Food,redmart,5518,5010,4247,889
6,Dairy,coldstorage,512,446,362,94
7,Dairy,fairprice,830,786,665,137
8,Dairy,redmart,2937,2825,2110,746
9,Drinks,coldstorage,1075,1040,905,142


## 18. Create balanced coverage table for pilot-category selection

In [18]:
catalog_pivot = (
    category_retailer_coverage
    .pivot(
        index="target_category_final",
        columns="retailer",
        values="catalog_eligible_rows",
    )
    .reindex(
        index=TARGET_CATEGORIES,
        columns=[
            "redmart",
            "fairprice",
            "coldstorage",
        ],
        fill_value=0
    )
    .fillna(0)
    .astype(int)
)

offer_pivot = (
    category_retailer_coverage
    .pivot(
        index="target_category_final",
        columns="retailer",
        values="offer_eligible_rows",
    )
    .reindex(
        index=TARGET_CATEGORIES,
        columns=[
            "redmart",
            "fairprice",
            "coldstorage",
        ],
        fill_value=0
    )
    .fillna(0)
    .astype(int)
)

pilot_coverage = pd.DataFrame(
    index=TARGET_CATEGORIES
)

pilot_coverage[
    "redmart_catalog"
] = catalog_pivot["redmart"]

pilot_coverage[
    "fairprice_catalog"
] = catalog_pivot["fairprice"]

pilot_coverage[
    "coldstorage_catalog"
] = catalog_pivot["coldstorage"]

pilot_coverage[
    "catalog_total"
] = (
    catalog_pivot.sum(axis=1)
)

pilot_coverage[
    "catalog_min_per_retailer"
] = (
    catalog_pivot.min(axis=1)
)

pilot_coverage[
    "catalog_retailer_coverage"
] = (
    (catalog_pivot > 0)
    .sum(axis=1)
)

pilot_coverage[
    "redmart_offer"
] = offer_pivot["redmart"]

pilot_coverage[
    "fairprice_offer"
] = offer_pivot["fairprice"]

pilot_coverage[
    "coldstorage_offer"
] = offer_pivot["coldstorage"]

pilot_coverage[
    "offer_total"
] = (
    offer_pivot.sum(axis=1)
)

pilot_coverage[
    "offer_min_per_retailer"
] = (
    offer_pivot.min(axis=1)
)

pilot_coverage[
    "offer_retailer_coverage"
] = (
    (offer_pivot > 0)
    .sum(axis=1)
)

pilot_coverage = (
    pilot_coverage
    .reset_index()
    .rename(
        columns={
            "index": "target_category"
        }
    )
)

display(
    pilot_coverage.sort_values(
        [
            "catalog_retailer_coverage",
            "catalog_min_per_retailer",
            "catalog_total",
        ],
        ascending=[
            False,
            False,
            False,
        ]
    )
)


,target_category,redmart_catalog,fairprice_catalog,coldstorage_catalog,catalog_total,catalog_min_per_retailer,catalog_retailer_coverage,redmart_offer,fairprice_offer,coldstorage_offer,offer_total,offer_min_per_retailer,offer_retailer_coverage
5,Snacks,10387,3547,1262,15196,1262,3,7280,3181,1090,11551,1090,3
1,Drinks,6219,2145,1040,9404,1040,3,4489,1989,905,7383,905,3
4,Canned & Packaged Food,5010,1357,628,6995,628,3,4247,1265,561,6073,561,3
2,Dairy,2825,786,446,4057,446,3,2110,665,362,3137,362,3
0,Rice & Noodles,2340,1877,357,4574,357,3,1791,1774,340,3905,340,3
7,Toiletries & Personal Care,1078,2932,284,4294,284,3,969,2775,248,3992,248,3
3,Breakfast Foods,951,333,248,1532,248,3,635,314,219,1168,219,3
6,Household Cleaning,2234,1366,38,3638,38,3,1752,1299,32,3083,32,3


## 19. Select pilot categories automatically

The default is three pilot categories.

Priority:

1. all three retailers represented;
2. at least 20 catalog-eligible records per retailer;
3. larger minimum retailer count;
4. larger total clean catalog count.

The threshold is configurable.


In [19]:
PILOT_CATEGORY_COUNT = 3
MIN_CATALOG_PER_RETAILER_FOR_PILOT = 20

pilot_ranked = (
    pilot_coverage
    .copy()
)

pilot_ranked[
    "meets_default_pilot_threshold"
] = (
    (
        pilot_ranked[
            "catalog_retailer_coverage"
        ]
        == 3
    )
    &
    (
        pilot_ranked[
            "catalog_min_per_retailer"
        ]
        >= MIN_CATALOG_PER_RETAILER_FOR_PILOT
    )
)

pilot_ranked = (
    pilot_ranked
    .sort_values(
        [
            "meets_default_pilot_threshold",
            "catalog_retailer_coverage",
            "catalog_min_per_retailer",
            "offer_min_per_retailer",
            "catalog_total",
        ],
        ascending=[
            False,
            False,
            False,
            False,
            False,
        ]
    )
    .reset_index(drop=True)
)

pilot_categories = (
    pilot_ranked
    .head(PILOT_CATEGORY_COUNT)
    .copy()
)

pilot_categories[
    "pilot_rank"
] = range(
    1,
    len(pilot_categories) + 1
)

print("Selected pilot categories:")
display(
    pilot_categories[
        [
            "pilot_rank",
            "target_category",
            "catalog_retailer_coverage",
            "catalog_min_per_retailer",
            "catalog_total",
            "offer_retailer_coverage",
            "offer_min_per_retailer",
            "offer_total",
            "meets_default_pilot_threshold",
        ]
    ]
)


Selected pilot categories:


,pilot_rank,target_category,catalog_retailer_coverage,catalog_min_per_retailer,catalog_total,offer_retailer_coverage,offer_min_per_retailer,offer_total,meets_default_pilot_threshold
0,1,Snacks,3,1262,15196,3,1090,11551,True
1,2,Drinks,3,1040,9404,3,905,7383,True
2,3,Canned & Packaged Food,3,628,6995,3,561,6073,True


## 20. Create pilot candidate pool

In [20]:
selected_pilot_categories = (
    pilot_categories[
        "target_category"
    ]
    .tolist()
)

pilot_candidate_pool = stage2a.loc[
    stage2a[
        "target_category_final"
    ].isin(
        selected_pilot_categories
    )
    &
    stage2a[
        "catalog_eligible"
    ].fillna(False)
].copy()

print(
    "Pilot candidate pool:",
    f"{len(pilot_candidate_pool):,}"
)

print(
    "\nBy pilot category and retailer:"
)

display(
    pd.crosstab(
        pilot_candidate_pool[
            "target_category_final"
        ],
        pilot_candidate_pool[
            "retailer"
        ],
        margins=True,
    )
)


Pilot candidate pool: 31,595

By pilot category and retailer:


retailer,coldstorage,fairprice,redmart,All
target_category_final,,,,
Canned & Packaged Food,628,1357,5010,6995
Drinks,1040,2145,6219,9404
Snacks,1262,3547,10387,15196
All,2930,7049,21616,31595


## 21. Verify out-of-stock catalog records remain in the pilot pool

If they fall within selected pilot categories and are otherwise catalog-eligible, they are retained.

They remain distinguishable through `currently_available` and `offer_eligible`.


In [21]:
pilot_out_of_stock = (
    pilot_candidate_pool[
        "stock_status"
    ]
    == "OUT_OF_STOCK"
)

print(
    "Pilot catalog records currently out of stock:",
    f"{int(pilot_out_of_stock.sum()):,}"
)

if pilot_out_of_stock.any():
    assert (
        pilot_candidate_pool.loc[
            pilot_out_of_stock,
            "catalog_eligible"
        ]
        .eq(True)
        .all()
    )

    assert (
        pilot_candidate_pool.loc[
            pilot_out_of_stock,
            "offer_eligible"
        ]
        .eq(False)
        .all()
    )

print(
    "PASS: Pilot selection preserves searchable "
    "out-of-stock listings."
)


Pilot catalog records currently out of stock: 6,588
PASS: Pilot selection preserves searchable out-of-stock listings.


## 22. Build a compact pilot matching table

In [22]:
pilot_matching_table = pilot_candidate_pool[
    [
        "target_category_final",
        "retailer",
        "source_record_id",
        "product_name_raw",
        "product_name_normalized",
        "brand_raw",
        "brand_normalized",
        "specification_raw",
        "pack_count",
        "unit_size",
        "unit",
        "total_quantity",
        "total_quantity_base",
        "base_unit",
        "product_price",
        "stock_status",
        "currently_available",
        "offer_eligible",
        "product_url_raw",
    ]
].copy()

display(
    pilot_matching_table.head(30)
)


,target_category_final,retailer,source_record_id,product_name_raw,product_name_normalized,brand_raw,brand_normalized,specification_raw,pack_count,unit_size,unit,total_quantity,total_quantity_base,base_unit,product_price,stock_status,currently_available,offer_eligible,product_url_raw
0,Snacks,redmart,redmart_00000002,Sojasun Natural Soya Yoghurt,sojasun natural soya yoghurt,Sojasun,sojasun,6 × 100 g,6,100.0,g,600.0,600.0,g,12.0,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i303336401-...
1,Snacks,redmart,redmart_00000003,Monoprix Chocolate Custard Dessert Cream 4x125G,monoprix chocolate custard dessert cream 4x125g,Monoprix,monoprix,4 × 125 g,4,125.0,g,500.0,500.0,g,10.9,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i462726209-...
2,Snacks,redmart,redmart_00000004,Top Gourmet Cheng Tng,top gourmet cheng tng,Top Gourmet,top gourmet,400 g,1,400.0,g,400.0,400.0,g,3.0,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i303250731-...
3,Snacks,redmart,redmart_00000005,La Laitiere Vanilla Rice Pudding Riz Au Lait 4...,la laitiere vanilla rice pudding riz au lait 4...,La Laitiere,la laitiere,4 × 115 g,4,115.0,g,460.0,460.0,g,10.09,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i301170176-...
4,Snacks,redmart,redmart_00000006,Monoprix Flan Caramel Vanilla Pudding 4X100G,monoprix flan caramel vanilla pudding 4x100g,Monoprix,monoprix,4 × 100 g,4,100.0,g,400.0,400.0,g,7.8,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i1711630409...
5,Snacks,redmart,redmart_00000007,Fortune Sweetened Chin Chow 1KG,fortune sweetened chin chow 1kg,Fortune,fortune,1 kg,1,1.0,kg,1.0,1000.0,g,2.85,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i301172144-...
6,Snacks,redmart,redmart_00000008,RedMart Trio Swiss Roll (Singapore Special),redmart trio swiss roll (singapore special),RedMart,redmart,430 g,1,430.0,g,430.0,430.0,g,10.1,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i1931806406...
7,Snacks,redmart,redmart_00000009,Arla Protein Chocolate Pudding 200G - Lactose ...,arla protein chocolate pudding 200g - lactose ...,Arla,arla,200 g,1,200.0,g,200.0,200.0,g,4.3,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i2769613302...
8,Snacks,redmart,redmart_00000010,Fortune Soya Beancurd Almond 360G,fortune soya beancurd almond 360g,Fortune,fortune,360 g,1,360.0,g,360.0,360.0,g,1.86,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i2366398545...
9,Snacks,redmart,redmart_00000011,Arla Protein Hazelnut Pudding 200G - Lactose Free,arla protein hazelnut pudding 200g - lactose free,Arla,arla,200 g,1,200.0,g,200.0,200.0,g,4.3,IN_STOCK,True,True,https://www.lazada.sg/products/pdp-i2769599294...


## 23. Validate Stage 2A row counts and provenance

In [23]:
assert len(stage2a) == EXPECTED_TOTAL_ROWS

assert (
    stage2a[
        "source_record_id"
    ].nunique()
    == EXPECTED_TOTAL_ROWS
)

assert (
    stage2a[
        "source_record_id"
    ].duplicated().sum()
    == 0
)

for retailer, expected_count in EXPECTED_RETAILER_COUNTS.items():
    actual_count = (
        stage2a[
            "retailer"
        ]
        == retailer
    ).sum()

    assert actual_count == expected_count

print(
    "PASS: Stage 2A preserved all rows "
    "and provenance IDs."
)


PASS: Stage 2A preserved all rows and provenance IDs.


## 24. Create source-category review summary

In [24]:
source_category_review = (
    source_category_inventory.loc[
        source_category_inventory[
            "mapping_status"
        ]
        != "AUTO_SOURCE_CATEGORY"
    ]
    .copy()
)

print(
    "Unique retailer/source-category combinations "
    "requiring review:",
    f"{len(source_category_review):,}"
)

display(
    source_category_review.head(50)
)


Unique retailer/source-category combinations requiring review: 1,316


,retailer,source_category_raw,source_category_normalized,total_rows,catalog_eligible_rows,offer_eligible_rows,out_of_stock_rows,auto_target_category,mapping_status
1667,redmart,Seasonal Accessories,seasonal accessories,1805,1693,1265,449,<NA>,REVIEW
1431,redmart,Fresh Herbs & Spices,fresh herbs & spices,1640,1553,1486,67,<NA>,REVIEW
1214,redmart,Baking Needs & Sugar,baking needs & sugar,1533,1518,1180,340,<NA>,REVIEW
1756,redmart,White Goods & Products,white goods & products,1246,1176,771,415,<NA>,REVIEW
1580,redmart,Multivitamin Supplements,multivitamin supplements,1201,1015,733,323,<NA>,REVIEW
1649,redmart,Red Products,red products,1073,996,995,2,<NA>,REVIEW
1307,redmart,Cooking Utensils,cooking utensils,1453,880,650,350,<NA>,REVIEW
1669,redmart,Sexual & Sensual Toys,sexual & sensual toys,899,879,471,416,<NA>,REVIEW
1418,redmart,Food Storage Savers,food storage savers,1101,814,513,381,<NA>,REVIEW
1671,redmart,Shampoos & Conditioners,shampoos & conditioners,851,782,587,204,<NA>,REVIEW


## 25. Save Stage 2A mapped dataset

In [25]:
STAGE2A_OUTPUT_FILE = (
    STAGING_DATA_DIR
    / "all_retailers_stage2a_category_mapped.csv"
)

stage2a.to_csv(
    STAGE2A_OUTPUT_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved Stage 2A mapped dataset:")
print(STAGE2A_OUTPUT_FILE)


Saved Stage 2A mapped dataset:
D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\staging\all_retailers_stage2a_category_mapped.csv


## 26. Save Stage 2A reports and pilot files

In [26]:
SOURCE_CATEGORY_INVENTORY_FILE = (
    CURATED_DATA_DIR
    / "stage2a_source_category_inventory.csv"
)

SOURCE_CATEGORY_REVIEW_FILE = (
    CURATED_DATA_DIR
    / "stage2a_source_category_review.csv"
)

MAPPING_REVIEW_FILE = (
    CURATED_DATA_DIR
    / "stage2a_product_mapping_review.csv"
)

CATEGORY_COVERAGE_FILE = (
    CURATED_DATA_DIR
    / "stage2a_category_retailer_coverage.csv"
)

PILOT_COVERAGE_FILE = (
    CURATED_DATA_DIR
    / "stage2a_pilot_coverage.csv"
)

PILOT_CATEGORIES_FILE = (
    CURATED_DATA_DIR
    / "stage2a_pilot_categories.csv"
)

PILOT_POOL_FILE = (
    CURATED_DATA_DIR
    / "stage2a_pilot_candidate_pool.csv"
)

PILOT_MATCHING_FILE = (
    CURATED_DATA_DIR
    / "stage2a_pilot_matching_table.csv"
)

source_category_inventory.to_csv(
    SOURCE_CATEGORY_INVENTORY_FILE,
    index=False,
    encoding="utf-8"
)

source_category_review.to_csv(
    SOURCE_CATEGORY_REVIEW_FILE,
    index=False,
    encoding="utf-8"
)

mapping_review_queue.to_csv(
    MAPPING_REVIEW_FILE,
    index=False,
    encoding="utf-8"
)

category_retailer_coverage.to_csv(
    CATEGORY_COVERAGE_FILE,
    index=False,
    encoding="utf-8"
)

pilot_coverage.to_csv(
    PILOT_COVERAGE_FILE,
    index=False,
    encoding="utf-8"
)

pilot_categories.to_csv(
    PILOT_CATEGORIES_FILE,
    index=False,
    encoding="utf-8"
)

pilot_candidate_pool.to_csv(
    PILOT_POOL_FILE,
    index=False,
    encoding="utf-8"
)

pilot_matching_table.to_csv(
    PILOT_MATCHING_FILE,
    index=False,
    encoding="utf-8"
)

print("Saved:")
for file_path in [
    SOURCE_CATEGORY_INVENTORY_FILE,
    SOURCE_CATEGORY_REVIEW_FILE,
    MAPPING_REVIEW_FILE,
    CATEGORY_COVERAGE_FILE,
    PILOT_COVERAGE_FILE,
    PILOT_CATEGORIES_FILE,
    PILOT_POOL_FILE,
    PILOT_MATCHING_FILE,
]:
    print("-", file_path)


Saved:
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_source_category_inventory.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_source_category_review.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_product_mapping_review.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_category_retailer_coverage.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_pilot_coverage.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_pilot_categories.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_pilot_candidate_pool.csv
- D:\DSML 2025\Semester 1 (2026)\CS5224\Project\GitHub\CartOptix\CartOptix\data\curated\stage2a_pilot_matching_table.csv


## 27. Verify Stage 2A output files

In [27]:
OUTPUT_FILES = [
    STAGE2A_OUTPUT_FILE,
    SOURCE_CATEGORY_INVENTORY_FILE,
    SOURCE_CATEGORY_REVIEW_FILE,
    MAPPING_REVIEW_FILE,
    CATEGORY_COVERAGE_FILE,
    PILOT_COVERAGE_FILE,
    PILOT_CATEGORIES_FILE,
    PILOT_POOL_FILE,
    PILOT_MATCHING_FILE,
]

for file_path in OUTPUT_FILES:
    print(
        f"{file_path.name:48s} -> "
        f"{file_path.exists()}"
    )

    assert file_path.exists()

print(
    "\nPASS: All Stage 2A output files were created."
)


all_retailers_stage2a_category_mapped.csv        -> True
stage2a_source_category_inventory.csv            -> True
stage2a_source_category_review.csv               -> True
stage2a_product_mapping_review.csv               -> True
stage2a_category_retailer_coverage.csv           -> True
stage2a_pilot_coverage.csv                       -> True
stage2a_pilot_categories.csv                     -> True
stage2a_pilot_candidate_pool.csv                 -> True
stage2a_pilot_matching_table.csv                 -> True

PASS: All Stage 2A output files were created.


## 28. Final Stage 2A validation

In [28]:
auto_mapped_catalog_count = int(
    (
        stage2a[
            "target_category_final"
        ].notna()
        &
        stage2a[
            "catalog_eligible"
        ].fillna(False)
    ).sum()
)

auto_mapped_offer_count = int(
    (
        stage2a[
            "target_category_final"
        ].notna()
        &
        stage2a[
            "offer_eligible"
        ].fillna(False)
    ).sum()
)

print("=" * 84)
print("STAGE 2A FINAL VALIDATION")
print("=" * 84)

print(
    f"Total records                       : "
    f"{len(stage2a):,}"
)

print(
    f"Unique source IDs                   : "
    f"{stage2a['source_record_id'].nunique():,}"
)

print(
    f"Duplicate source IDs                : "
    f"{stage2a['source_record_id'].duplicated().sum():,}"
)

print("-" * 84)

print(
    f"Auto source-category mapped         : "
    f"{auto_mapped_count:,}"
)

print(
    f"Product-text suggestions            : "
    f"{product_suggestion_count:,}"
)

print(
    f"Mapping conflicts                   : "
    f"{conflict_count:,}"
)

print(
    f"Unmapped review                     : "
    f"{unmapped_count:,}"
)

print("-" * 84)

print(
    f"Auto-mapped catalog-eligible rows   : "
    f"{auto_mapped_catalog_count:,}"
)

print(
    f"Auto-mapped active offer rows       : "
    f"{auto_mapped_offer_count:,}"
)

print("-" * 84)

print(
    f"Selected pilot categories           : "
    f"{len(pilot_categories):,}"
)

for row in pilot_categories[
    [
        "pilot_rank",
        "target_category",
    ]
].itertuples(index=False):
    print(
        f"  {row.pilot_rank}. "
        f"{row.target_category}"
    )

print(
    f"Pilot candidate pool                : "
    f"{len(pilot_candidate_pool):,}"
)

print(
    f"Pilot out-of-stock catalog records  : "
    f"{int(pilot_out_of_stock.sum()):,}"
)

print("=" * 84)

assert len(stage2a) == 177_638

assert (
    stage2a[
        "source_record_id"
    ].nunique()
    == 177_638
)

assert (
    stage2a[
        "source_record_id"
    ].duplicated().sum()
    == 0
)

assert set(
    stage2a[
        "target_category_final"
    ]
    .dropna()
    .unique()
) <= set(TARGET_CATEGORIES)

assert len(
    pilot_categories
) <= PILOT_CATEGORY_COUNT

assert set(
    selected_pilot_categories
) <= set(TARGET_CATEGORIES)

assert stage2a[
    STAGE1E_COLUMNS
].equals(stage1e_snapshot)

print()
print(
    "PASS: Stage 2A category mapping and "
    "pilot-category selection succeeded."
)

print()
print(
    "Important: only unique source-category rule matches "
    "are automatically accepted. Product-name mappings "
    "remain review suggestions."
)


STAGE 2A FINAL VALIDATION
Total records                       : 177,638
Unique source IDs                   : 177,638
Duplicate source IDs                : 0
------------------------------------------------------------------------------------
Auto source-category mapped         : 55,787
Product-text suggestions            : 10,197
Mapping conflicts                   : 2,542
Unmapped review                     : 109,112
------------------------------------------------------------------------------------
Auto-mapped catalog-eligible rows   : 49,690
Auto-mapped active offer rows       : 40,292
------------------------------------------------------------------------------------
Selected pilot categories           : 3
  1. Snacks
  2. Drinks
  3. Canned & Packaged Food
Pilot candidate pool                : 31,595
Pilot out-of-stock catalog records  : 6,588

PASS: Stage 2A category mapping and pilot-category selection succeeded.

Important: only unique source-category rule matches are automa

## Stage 2A result

When the final validation passes:

- all Stage 1E rows and provenance remain unchanged;
- the eight CartOptix target categories are explicitly defined;
- obvious retailer source categories are mapped automatically;
- ambiguous and unmatched categories remain reviewable;
- product-name matching is used only as a suggestion;
- category coverage is measured separately for searchable catalog records and active in-stock offers;
- up to three categories are selected for the cross-retailer pilot using balanced coverage;
- out-of-stock but catalog-eligible records remain in the pilot candidate pool.

### Next stage

Proceed to **Stage 2B: cross-retailer candidate generation and exact product matching for the selected pilot categories**.

Stage 2B should compare:

- normalized brand;
- product name / variant evidence;
- pack count;
- unit size / quantity;
- total base quantity;
- retailer availability;

while ensuring that fuzzy similarity only proposes candidates and never makes the final verification decision.
